In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.model_selection import train_test_split,KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import f_regression


In [ ]:
df = pd.read_csv("expenses.csv")


df.info()
df.describe()
df.isnull().sum()
df.nunique()

In [ ]:
df.hist(bins=30, figsize=(12,10))
plt.show()



In [ ]:
df_encoded = pd.get_dummies(df,drop_first=True)
plt.figure(figsize=(10,8))
sns.heatmap(df_encoded.corr(),annot=True , cmap="coolwarm")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(x="smoker",y="charges",data=df)
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(x="sex",y="charges",data=df)
plt.show()

In [ ]:
sns.pairplot(df,hue="smoker")
plt.show()

In [ ]:
Q1 = df['charges'].quantile(0.25)
Q3 = df['charges'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

outliers = df[(df['charges'] < lower) | (df['charges'] > upper)]
print("Outliers in charges:", len(outliers))

In [ ]:
# One-hot encode categorical columns
df_encoded = pd.get_dummies(df, drop_first=True)

# Scale numeric columns
scaler = StandardScaler()
num_cols = ['age', 'bmi', 'children']
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])

# Convert boolean columns to int (0/1)
df_encoded = df_encoded.astype({col: int for col in df_encoded.select_dtypes('bool').columns})

# Log-transform target
df_encoded["charges_log"] = np.log1p(df_encoded["charges"])

In [ ]:
# Correlation with target
corr = df_encoded.corr()['charges_log'].sort_values(ascending=False)
print("\nCorrelation with log(charges):\n", corr)

# Random Forest feature importance
X = df_encoded.drop(["charges", "charges_log"], axis=1)
y = df_encoded["charges_log"]

rf = RandomForestRegressor(random_state=42)
rf.fit(X, y)

feat_importances = pd.Series(rf.feature_importances_, index=X.columns)
feat_importances.sort_values().plot(kind="barh", figsize=(8,6))
plt.title("Random Forest Feature Importance")
plt.show()

# ANOVA F-test
f_scores, p_values = f_regression(X, y)
anova = pd.DataFrame({"Feature": X.columns, "F_Score": f_scores, "p_value": p_values})
print("\nANOVA F-test Results:\n", anova.sort_values(by="F_Score", ascending=False))


In [ ]:
# Final dataset checks
print("\nHead of dataset:\n", df_encoded.head())
print("\nTail of dataset:\n", df_encoded.tail())
print("\nShape:", df_encoded.shape)
print("\nInfo:\n")
print(df_encoded.info())
print("\nDescribe:\n", df_encoded.describe())


In [ ]:
X = df_encoded.drop(["charges", "charges_log"], axis=1)
y = df_encoded["charges"]

# 80/20 split (same as IJERPH 2022 & Math 2023 papers)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


In [ ]:
# Example: simulate new test cases (not in dataset)
new_data = pd.DataFrame({
    "age": [25, 60],
    "bmi": [22.5, 35.0],
    "children": [0, 2],
    "sex_male": [1, 0],
    "smoker_yes": [0, 1],
    "region_northwest": [0, 0],
    "region_southeast": [1, 0],
    "region_southwest": [0, 1]
})

# Apply scaling to new_data numeric cols
new_data[num_cols] = scaler.transform(new_data[num_cols])
print(new_data)
